
# 2 - Erstellung der info.json

Dieses Script erstellt eine info.json Datei für alle Objekte der Collection und legt sie im Ordner 'info' als JSON Datei ab.
Doku der Info.json, siehe GOCFL implementierung von Jürgen Enge, https://github.com/je4/gocfl . 
Vorlage: https://github.com/je4/gocfl/blob/main/gocfl-info-1.0.json 


## Eingabedatei

Die Excel-Datei liegt im working directory. Sie kann relativ leicht aus Alma exportiert werden. Die Sammlungen der Sosa sind alle in Alma in einem öffentlichen Set in der RZS gelistet. Bsp. E-Manuscripta:

    zhb_e-manuscripta

Die Export-Datei wurde leicht überarbeitet. Nicht benötigte Spalten werden gelöscht, einige Daten müssen gesplitted werden. Folgende Spalten werden benötigt:

- Title
- Record number: wird vorerst nicht benötigt, kann trotzdem stehengelassen werden.
- Call number: aus Spalte Availability splitten, Spalte umbenennen 
- MMS_ID als Text erzwingen
- DOI händisch ergänzen
- Dateipfad händisch ergänzen
- externe ID wie z.B. E-Manuscripta ID händisch ergänzen

Da die Sosa-Sammlungen der ZHB i.d.R. überschaubar sind, hält sich der zeitliche Aufwand dafür in Grenzen.


## Export info.json

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format info/{signature}.json.
Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert ins directory 'files' als menschenlesbarer Nachweis, welche Datenobjekte eingelagert wurden. Auch eine Liste aller Signaturen wird als Text-Datei dort abgelegt.
Die Signaturendatei ist so konfiguriert, dass 

In [ ]:
import json
import pandas as pd
import config

from datetime import datetime


# needed variables: files, paths, input

input_file = config.input_file
collection = config.collection_id
info_dir = f'{collection}/{config.info_path}/'
files_dir = f'{collection}/{config.files_path}/{collection}'
urn = config.ingest_workflow

sigfile = f"{files_dir}_signatures.txt"
fulljsonfile = f"{files_dir}_complete_set.json"
fullexcelfile = f"{files_dir}_complete_set.xlsx"

sigList = []
completeSet = []

today = datetime.today().strftime('%Y-%m-%d')
now = datetime.now().isoformat()
doc_counter = 0

# Read the Excel file into a pandas DataFrame
df = pd.read_excel(input_file)

for _, row in df.iterrows():

    doc_counter += 1
    infoSet = {
    # infoset created after https://github.com/je4/gocfl/blob/main/gocfl-info-1.0.json 

        "signature": "",
        "organisation_id": config.organisation_id,
        "organisation": config.organisation,
        "collection_id": config.collection_id,
        "collection": config.collection,
        "sets": config.sets,
        "identifiers": [],
        "title": "",
        "alternative_titles": [],
        "description": "",
        "keywords": config.keywords,
        "user": config.user_name,
        "address": config.user_address,
        "created": now,
        "last_changed": now,
        "deprecates": "",
        "references": [],
        "ingest_workflow": config.ingest_workflow,
        "additional": ""
    }

    
    # identifiers
        
    doi = row['DOI']
    mms_id = str(row['MMS ID'])
    callnumber = row['Call number']
    title = row['Title']
    aip_path = row['Dateipfad']
    external_id = row['externe ID']
    urn_external_id = f"{urn}:{external_id}"
    
    # folder name and signature:
    foldername = doi.replace('.','_').replace('/','_')
    signature = config.organisation_id+'_'+foldername
    
    # references
    doiurl = config.baseurl_doi+doi
    almaurl = config.baseurl_alma+mms_id
    
    #complete info.json
    
    infoSet["identifiers"] = ['doi:'+doi, 'mmsid:'+mms_id, 'zhb:'+callnumber, urn_external_id]
    infoSet["references"] = [doiurl, almaurl]
    infoSet["signature"] = signature
    infoSet["title"] = title
    infoSet["additional"] = aip_path.replace('\\','/')

    #print(infoSet)
    
    completeSet.append(infoSet)
    sigList.append(signature)
    
    # Write the infoSet to a JSON file
    info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
    infofile = f"{info_dir}/{signature}.json"

    with open(infofile, "w") as outfile:
        outfile.write(info_json)
        print(f"info.json saved as {infofile}")
        
'''    # write signature to file
    with open(sigfile, 'a') as file:
        file.write(signature)
        file.write("\n")
        print(f"{doc_counter}. signature appended to {sigfile}\n")
'''
# write sigList to textfile
with open(sigfile, 'w') as f:
    for sig in sigList:
        f.write(f"{sig}\n")
        
print(f"\n{doc_counter} signatures printed to {sigfile}\n")

# Writing completeSet as json file
fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)

with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nAll JSON written to {fulljsonfile}")
    
# Writing completeSet as Excel file

df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"All data saved to Excel file as {fullexcelfile}")
print(f"Total records: {doc_counter}. Finished at {now}")


